# Reproducing one target end-to-end: DrugCLIP + MaxSim reranker

This notebook takes a single held-out DUD-E target (`akt1`) from DrugCLIP's shortlist
through to the final reranked ordering, using the **frozen** DrugCLIP checkpoint and a
**trained linear projection** W.

Pipeline: load checkpoint -> train/load W (layer 12) -> Stage-1 dot ranking ->
top-1000 shortlist -> Stage-2 MaxSim rerank -> cascade (rerank shortlist, rest keep dot
order) -> enrichment.

Run from `~/drugclip/DrugCLIP` in the `drugclip` conda env.


## 1. Load the frozen DrugCLIP model

In [ ]:
import os, sys, random
import numpy as np, torch
import torch.nn.functional as F
from unicore import options, tasks, checkpoint_utils, distributed_utils
import unicore

# Build the unicore args exactly as the CLI scripts do, then add the W hyperparameters.
parser = options.get_validation_parser()
parser.add_argument('--dude-root', type=str,
    default=os.path.expanduser('~/drugclip/dude_data/data/protein/DUD-E/raw/all'))
options.add_model_args(parser)
argv = ['--user-dir','./unimol','./data','--valid-subset','test','--task','drugclip',
        '--loss','in_batch_softmax','--arch','drugclip','--seed','1',
        '--path','../checkpoint_best.pt','--max-pocket-atoms','511',
        '--batch-size','8','--num-workers','2','--ddp-backend=c10d']
sys.argv = ['notebook'] + argv
args = options.parse_args_and_arch(parser)
for k, v in dict(decoys_per_target=800, neg_per_step=32, hard_pool=96, steps=4000,
                 proj_dim=128, dropout=0.1, proj_wd=1e-2, tau=0.05,
                 share_w=False, random_neg=False, n_test=20, n_val=10, n_train=60).items():
    setattr(args, k, v)

random.seed(1); np.random.seed(1); torch.manual_seed(1)
state = checkpoint_utils.load_checkpoint_to_cpu(args.path)
task = tasks.setup_task(args)
model = task.build_model(args); model.load_state_dict(state['model'], strict=False)
use_cuda = torch.cuda.is_available()
if use_cuda: model.cuda()
model.eval()
dev = next(model.parameters()).device
full = len(model.mol_model.encoder.layers)
LAYER, ALPHA = 12, 0.5
print('model loaded on', dev, '| layers', full, '| token layer', LAYER)

## 2. The trained projection W
W (two linear maps, 512->128) is the only trained part. We load a cached copy if
present, otherwise train it on the 60 DUD-E training targets (seed 1, ~10-15 min) and
cache it. The encoders stay frozen throughout.

In [ ]:
from rerank_eval import train_W, encode_target, maxsim

targets = sorted(os.listdir(args.dude_root))
train_ts = targets[args.n_test + args.n_val: args.n_test + args.n_val + args.n_train]
W_PATH = os.path.expanduser('~/W_layer12.pt')

Wm = torch.nn.Linear(512, 128, bias=False).to(dev)
Wp = torch.nn.Linear(512, 128, bias=False).to(dev)
if os.path.exists(W_PATH):
    sd = torch.load(W_PATH, map_location=dev)
    Wm.load_state_dict(sd['Wm']); Wp.load_state_dict(sd['Wp'])
    print('loaded cached W from', W_PATH)
else:
    print('training W on', len(train_ts), 'DUD-E targets ...')
    Wm, Wp = train_W(args, task, model, dev, use_cuda, full, LAYER, train_ts)
    torch.save({'Wm': Wm.state_dict(), 'Wp': Wp.state_dict()}, W_PATH)
    print('saved W to', W_PATH)
Wm.eval(); Wp.eval()

## 3. Stage 1 - DrugCLIP dot ranking and the shortlist
Encode the target's pocket and every library molecule, score each by the pooled
dot product (DrugCLIP), and keep the top-1000 as the shortlist.

In [ ]:
TARGET = 'akt1'
p_atoms, pcls, atoms, labels, clss = encode_target(task, model, args.dude_root,
                                                   TARGET, use_cuda, None, LAYER, full)
labels = np.asarray(labels)
dot = (F.normalize(torch.stack(clss), dim=-1) @ pcls.T).squeeze(1).cpu().numpy()
n_mols, n_act = len(dot), int(labels.sum())
K = 1000
order_dot = np.argsort(-dot, kind='mergesort')
short = order_dot[:K]
print(f'{TARGET}: {n_mols} molecules, {n_act} actives ({100*n_act/n_mols:.2f}%)')
print(f'shortlist (top-{K} by dot) contains {int(labels[short].sum())} actives '
      f'-> Recall@{K} = {labels[short].sum()/n_act:.3f} (ceiling for any reranker)')

## 4. Stage 2 - MaxSim rerank of the shortlist
For each shortlisted molecule, compute the trained MaxSim score against the pocket,
combine with the dot via `z(dot) + alpha * z(maxsim)`, and rerank. Molecules below the
shortlist keep their dot order (the deployment cascade).

In [ ]:
def z(x):
    x = np.asarray(x, float); return (x - x.mean()) / (x.std() + 1e-8)

ms = np.array([maxsim(atoms[i].to(dev), p_atoms, Wm, Wp).item() for i in short])
combined = z(dot[short]) + ALPHA * z(ms)              # rerank score on the shortlist
short_reranked = short[np.argsort(-combined, kind='mergesort')]
final_order = np.concatenate([short_reranked, order_dot[K:]])  # cascade
print('reranked the', K, 'shortlisted molecules; built final cascade ranking')

## 5. Enrichment: DrugCLIP dot vs the reranked cascade
Early-recognition metrics on the full ranking, plus actives recovered in the very top.

In [ ]:
from unimol.tasks.drugclip import cal_metrics

def rank_score(order):
    s = np.empty(len(order)); s[order] = np.arange(len(order), 0, -1); return s

def report(name, order):
    s = rank_score(order)
    auc, bed, ef, _ = cal_metrics(labels, s, 80.5)
    top = order[:100]
    a50 = int(labels[order[:50]].sum()); a100 = int(labels[top].sum())
    print(f'{name:16s} BEDROC {bed:.4f}  EF@0.5%% {ef["0.005"]:6.2f}  '
          f'EF@1%% {ef["0.01"]:6.2f}  actives@50 {a50:3d}  actives@100 {a100:3d}')
    return bed, ef['0.005'], a50, a100

print('target:', TARGET, '\n')
b0 = report('DrugCLIP dot', order_dot)
b1 = report('dot + MaxSim', final_order)
print(f'\nΔBEDROC {b1[0]-b0[0]:+.4f}   ΔEF@0.5%% {b1[1]-b0[1]:+.2f}   '
      f'Δactives@50 {b1[2]-b0[2]:+d}   Δactives@100 {b1[3]-b0[3]:+d}')

## 6. Cumulative actives in the top of the ranking (optional plot)

In [ ]:
N = 1000
cum_dot = np.cumsum(labels[order_dot[:N]])
cum_rr  = np.cumsum(labels[final_order[:N]])
print(f'actives recovered by rank {N}:  dot {cum_dot[-1]}   reranked {cum_rr[-1]}')
try:
    import matplotlib; matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6,4))
    plt.plot(range(1, N+1), cum_dot, label='DrugCLIP dot', lw=2)
    plt.plot(range(1, N+1), cum_rr,  label='dot + MaxSim (reranked)', lw=2)
    plt.xlabel('rank'); plt.ylabel('cumulative actives recovered')
    plt.title(f'{TARGET}: early recovery, dot vs reranked'); plt.legend(); plt.tight_layout()
    plt.savefig(os.path.expanduser('~/repro_akt1_recovery.png'), dpi=120)
    print('saved plot to ~/repro_akt1_recovery.png'); plt.show()
except ModuleNotFoundError:
    print('(matplotlib not installed - skipping plot; the counts above are the result)')

## 7. Takeaways
- The trained MaxSim reranker improves early enrichment on this held-out target at
  negligible added cost (MaxSim ~0.013 ms/mol; the real cost is re-encoding the shortlist).
- End-to-end recovery is capped by the Stage-1 Recall@1000 shown above.
- **Scope note.** This notebook demonstrates the *shortlist cascade*: Stage 2 reranks only
  the top-1000 and z-scores within it, so the counts here are the cascade-bounded version.
  The paper's full-library rerank (Table 1 and Appendix C) gives a slightly larger per-target
  gain because it can also promote actives from below the shortlist; e.g. for akt1 this
  notebook reports actives@50 12->27, while the full-library Table gives 12->31.
- **Caveat (specificity):** the companion `pocket_subst.py` control shows that on DUD-E
  this reranking gain is largely *not pocket-specific* (a wrong pocket, and a ligand-only
  head, rerank about as well), consistent with DUD-E's known decoy bias. Swap the pocket
  in Stage 2 (use another target's `p_atoms`) to see this directly.